In [17]:
!pip install beautifulsoup4

In [18]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re

In [19]:
url='https://www.amazon.in/s?k=mobile&crid=123WVWCM9RCED&sprefix=mobil%2Caps%2C363&ref=nb_sb_noss_2'

In [20]:
headers={'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/145.0.0.0 Safari/537.36 Edg/145.0.0.0'}

In [41]:
data=[]

In [42]:
responce=requests.get(url,headers=headers)
responce.status_code

200

In [43]:
for page in range(1, 5):
    params = {"k": "laptop", "page": page}

    response = requests.get(url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    products = soup.find_all("div", {"data-component-type": "s-search-result"})

    for product in products:
        title_tag = product.find("h2")
        if not title_tag:
            continue

        title_text = title_tag.get_text(strip=True)

        price_tag = product.find("span", {"class": "a-price-whole"})
        price_text = price_tag.get_text(strip=True) if price_tag else "N/A"

        match = re.match(r'^\W*([A-Za-z]+)', title_text)
        brand = match.group(1).upper() if match else "UNKNOWN"

        ram_match = re.search(r'(\d+)\s*GB\s*(RAM|DDR\d+|LPDDR\d+)', title_text, re.IGNORECASE)
        ram = ram_match.group(1) + "GB" if ram_match else "N/A"

        rating_tag = product.find("span", class_="a-icon-alt")
        rating = "N/A"
        if rating_tag:
            rating_match = re.search(r'(\d+\.?\d*)', rating_tag.get_text())
            rating = rating_match.group(1) if rating_match else "N/A"

        ssd_match = re.search(r'(\d+)\s*(GB|TB)\s*(?:SSD|NVMe)', title_text, re.IGNORECASE)
        ssd = (ssd_match.group(1) + ssd_match.group(2).upper()) if ssd_match else "N/A"

        windows_match = re.search(r'(?:Windows\s*|Win\s*)(\d+)', title_text, re.IGNORECASE)
        windows_version = "Windows " + windows_match.group(1) if windows_match else "N/A"

        color_match = re.search(r'\b(Black|Silver|Gray|White|Blue|Red)\b', title_text, re.IGNORECASE)
        color = color_match.group(1) if color_match else "N/A"

        processor = "N/A"
        intel_match = re.search(r'Intel\s+(?:Core\s+)?(?:i[3579])[-\w]*', title_text, re.IGNORECASE)
        if intel_match:
            processor = intel_match.group(0)

        data.append({
            "Title": title_text,
            "Price": price_text,
            "Brand": brand,
            "RAM": ram,
            "Rating": rating,
            "Storage": ssd,
            "Windows": windows_version,
            "Color": color,
            "Processor": processor
        })


In [44]:
# make data into a dataframe 
import pandas as pd

# Create a DataFrame from the data list
df = pd.DataFrame(data)
df.head()

,Title,Price,Brand,RAM,Rating,Storage,Windows,Color,Processor
0,"Dell 15, Intel Core i5 13th Gen-1334U Processo...","56,990",DELL,16GB,3.7,512GB,Windows 11,N/A,Intel Core i5
1,"Dell SmartChoice G15-5530, Intel Core i5 13th ...","77,787",DELL,16GB,4.0,1TB,Windows 11,N/A,Intel Core i5
2,"BrowseBook 14.1"" FHD IPS Laptop | Best Student...","12,990",BROWSEBOOK,4GB,3.0,128GB,Windows 11,N/A,N/A
3,"EBook 11.6"" HD Laptop | Best Student & Office ...","10,990",EBOOK,4GB,4.0,N/A,Windows 11,Black,N/A
4,JioBook 11 with Lifetime Office | Android 4G L...,"10,990",JIOBOOK,4GB,2.9,N/A,N/A,Blue,N/A


In [45]:
# Save the DataFrame to a CSV file
df.to_csv("amazon_laptops.csv", index=False)